# Modelado Predictivo de la Infracción C29

Se desarrolla un modelo predictivo para la infracción C29 (exceso de velocidad) empleando una metodología iterativa en la que cada nuevo modelo aprende de los resultados obtenidos por los modelos previamente entrenados. El proceso inicia con el modelo Holt-Winters, cuyos parámetros se ajustan con base en los hallazgos del análisis exploratorio y la descomposición STL de la serie temporal. Posteriormente, se incorporan secuencialmente los modelos ARIMA, SARIMA, Dynamic Optimized Theta, Ridge, Lasso, Random Forest, XGBoost, LightGBM y KNN, de modo que cada uno aprovecha el conocimiento acumulado por sus predecesores, con el objetivo de mejorar progresivamente el rendimiento predictivo. Las métricas de evaluación utilizadas son RMSE, MAE, MAPE, SMAPE y MSE. Finalmente, se selecciona el mejor modelo para realizar pronósticos del volumen de infracciones por exceso de velocidad para el año 2026, cuyos datos aún no han sido publicados por la Alcaldía de Barranquilla a través de su portal de Datos Abiertos.

## Carga de librerías

Se importan las librerías necesarias para el desarrollo de modelos predictivos sobre las series temporales de comparendos electrónicos. Se emplean `pandas` y `numpy` para la manipulación y transformación de datos, `plotly.graph_objects` y `plotly.subplots` para la visualización de resultados, y `statsmodels.tsa` para los modelos estadísticos clásicos (Holt-Winters, ARIMA, SARIMA y STL). Para los enfoques de machine learning, se utilizan `scikit-learn` con `RidgeCV`, `MultiTaskLassoCV`, `RandomForestRegressor`, `KNeighborsRegressor` y `GridSearchCV` para optimización de hiperparámetros, junto con `xgboost` (XGBoost) y `lightgbm` (LightGBM). Adicionalmente, se incorporan `sklearn.model_selection.TimeSeriesSplit` para validación con series temporales, `sklearn.preprocessing.StandardScaler` para estandarización de características, y `sklearn.metrics` para las métricas de evaluación (RMSE, MAE, MAPE, SMAPE y MSE). Se suprimen los mensajes de advertencia para mantener una salida limpia y enfocada en los resultados.

In [52]:
import warnings

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from scipy import stats as sp_stats
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')

In [53]:
pio.renderers.default = "notebook_connected"

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [54]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [55]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [56]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [57]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [58]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

## Preparación de la Serie Temporal Mensual

Se extrae y configura la serie temporal mensual correspondiente a la infracción C29 (exceso de velocidad) a partir del DataFrame de comparendos electrónicos. Para ello, se agrupan los registros por mes y código de infracción, sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por período. La serie resultante se indexa con fechas mensuales, se establece una frecuencia mensual ('MS') y se rellenan los valores faltantes con cero, asegurando una secuencia temporal continua y completa para los modelos predictivos. Se imprime información básica de la serie: número de meses, rango temporal y total de comparendos del período.

In [59]:
df_comparendos_electronicos_copy = df_comparendos_electronicos.copy()
df_comparendos_electronicos_copy['año_mes'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.to_period('M').astype(str)

def preparar_serie_mensual(df, codigo_infraccion):
    infracciones_por_mes = df.groupby(['año_mes', 'COD_INFRACCION'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    
    serie = infracciones_por_mes[infracciones_por_mes['COD_INFRACCION'] == codigo_infraccion].copy()
    serie = serie.set_index('año_mes')['CANTIDAD_INFRACCIONES']
    
    serie.index = pd.to_datetime(serie.index)
    serie = serie.asfreq('MS')
    serie = serie.fillna(0)
    
    return serie

serie_c29 = preparar_serie_mensual(df_comparendos_electronicos_copy, 'C29')
print(f"Serie C29: {len(serie_c29)} meses")
print(f"Desde: {serie_c29.index.min().strftime('%Y-%m')} hasta: {serie_c29.index.max().strftime('%Y-%m')}")
print(f"Total comparendos: {serie_c29.sum():,.0f}")

Serie C29: 96 meses
Desde: 2018-01 hasta: 2025-12
Total comparendos: 284,755


## División de la Serie Temporal en Conjuntos de Entrenamiento y Prueba

Se particiona la serie temporal mensual de la infracción C29 en dos conjuntos: entrenamiento y prueba. El conjunto de entrenamiento comprende los meses desde enero de 2018 hasta diciembre de 2024, abarcando 84 meses, mientras que el conjunto de prueba corresponde al año completo de 2025, con 12 meses. Esta división permite entrenar los modelos predictivos con los datos históricos y evaluar su desempeño pronosticando el período más reciente (2025), cuyos valores reales ya están disponibles en la base de datos para calcular las métricas de error. Se imprime información detallada de ambos conjuntos, incluyendo su rango temporal y número de meses.

In [60]:
train_start = '2018-01-01'
train_end = '2024-12-01'
test_start = '2025-01-01'
test_end = '2025-12-01'

train_c29 = serie_c29[train_start:train_end].copy()
test_c29 = serie_c29[test_start:test_end].copy()

print(f"Train: {train_c29.index.min().strftime('%Y-%m')} a {train_c29.index.max().strftime('%Y-%m')} ({len(train_c29)} meses)")
print(f"Test: {test_c29.index.min().strftime('%Y-%m')} a {test_c29.index.max().strftime('%Y-%m')} ({len(test_c29)} meses)")

Train: 2018-01 a 2024-12 (84 meses)
Test: 2025-01 a 2025-12 (12 meses)


## Generación de Ventanas de Validación Cruzada Temporal

Se implementa una función de validación cruzada específica para series temporales, que respeta el orden cronológico de los datos y evita la fuga de información del futuro. La función `time_series_cv_manual` genera un conjunto de ventanas de entrenamiento y validación, donde el tamaño de cada ventana de validación es de 12 meses (un año completo). Se configuran 4 ventanas de validación, lo que significa que el modelo se evalúa con los últimos 4 años de datos disponibles. Para la infracción C29, se imprimen los rangos temporales de cada fold, mostrando el período de entrenamiento y validación correspondiente. Esta metodología permite evaluar el desempeño predictivo de los modelos en diferentes horizontes temporales de manera robusta y realista.

In [61]:
def time_series_cv_manual(serie, n_ventanas=4, horizonte=12):
    ventanas = []
    
    for i in range(n_ventanas, 0, -1):
        train_end_idx = len(serie) - (i * horizonte)
        train_fold = serie.iloc[:train_end_idx]
        val_fold = serie.iloc[train_end_idx:train_end_idx + horizonte]
        
        ventanas.append((train_fold, val_fold))
        
    return ventanas

ventanas_c29 = time_series_cv_manual(train_c29, n_ventanas=4, horizonte=12)

print("Ventanas de validación generadas:")
for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"Fold {i}: Train {train_fold.index.min().strftime('%Y-%m')} a {train_fold.index.max().strftime('%Y-%m')} "
          f"({len(train_fold)}m) -> Val {val_fold.index.min().strftime('%Y-%m')} a {val_fold.index.max().strftime('%Y-%m')} "
          f"({len(val_fold)}m)")

Ventanas de validación generadas:
Fold 1: Train 2018-01 a 2020-12 (36m) -> Val 2021-01 a 2021-12 (12m)
Fold 2: Train 2018-01 a 2021-12 (48m) -> Val 2022-01 a 2022-12 (12m)
Fold 3: Train 2018-01 a 2022-12 (60m) -> Val 2023-01 a 2023-12 (12m)
Fold 4: Train 2018-01 a 2023-12 (72m) -> Val 2024-01 a 2024-12 (12m)


## Holt-Winters

In [62]:
def evaluar_holt_amortiguado(train_fold, val_fold):
    """
    Ajusta un modelo Holt con tendencia aditiva y amortiguamiento,
    sin componente estacional, de acuerdo con los hallazgos de la
    descomposición: tendencia significativa decreciente y estacionalidad débil.
    El amortiguamiento busca capturar la posible atenuación de la tendencia
    y reducir la autocorrelación residual.
    """
    modelo = ExponentialSmoothing(
        train_fold,
        trend='add',          
        seasonal=None,        
        damped_trend=True,
        initialization_method='estimated'
    ).fit()
    
    predicciones = modelo.forecast(len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo

In [63]:
resultados_cv_c29 = []

print("-" * 50)
print("Validación cruzada - Holt con tendencia amortiguada (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo = evaluar_holt_amortiguado(train_fold, val_fold)
    
    resultados_cv_c29.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_c29 = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_c29]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_c29]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_c29]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_c29]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_c29])
}

print()
print("-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_c29['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_c29['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_c29['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_c29['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_c29['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Holt con tendencia amortiguada (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
RMSE: 684.10
MAE: 630.19
MAPE: 24.48%
SMAPE: 21.43%
MSE: 467986.82

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
RMSE: 791.99
MAE: 640.83
MAPE: 20.71%
SMAPE: 22.09%
MSE: 627242.40

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
RMSE: 934.46
MAE: 755.43
MAPE: 32.64%
SMAPE: 23.68%
MSE: 873207.65

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
RMSE: 760.57
MAE: 573.45
MAPE: 26.07%
SMAPE: 20.83%
MSE: 578469.75

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 792.78
MAE: 649.97
MAPE: 25.97%
SMAPE: 22.01%
MSE: 636726.65


In [64]:
print("-" * 50)
print("Evaluación final en Test (2025) - Holt con tendencia amortiguada")
print("-" * 50)

modelo_final_c29 = ExponentialSmoothing(
    train_c29,
    trend='add',
    seasonal=None,
    damped_trend=True,          
    initialization_method='estimated'
).fit()

predicciones_test_c29 = modelo_final_c29.forecast(len(test_c29))

real_test = test_c29.values
pred_test = predicciones_test_c29.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

metricas_test_c29 = {
    'MSE': mse_test,
    'RMSE': rmse_test,
    'MAE': mae_test,
    'MAPE': mape_test,
    'SMAPE': smape_test
}

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

--------------------------------------------------
Evaluación final en Test (2025) - Holt con tendencia amortiguada
--------------------------------------------------
RMSE: 479.43
MAE: 405.89
MAPE: 36.52%
SMAPE: 28.19%
MSE: 229851.11


In [65]:
meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index,
    y=train_c29.values,
    mode='lines+markers',
    name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=test_c29.values,
    mode='lines+markers',
    name='Test Real (2025)',
    line=dict(color='green', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=predicciones_test_c29.values,
    mode='lines+markers',
    name='Predicción Holt amortiguado',   
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2025-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2025-01-01',
    y=1, 
    yref='paper',
    text='<b>Inicio Test</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text=f'C29 - Holt con tendencia amortiguada: Predicción vs Real<br>'
             f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses,
    y=test_c29.values,
    mode='lines+markers',
    name='Real',
    line=dict(color='green', width=2),
    marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses,
    y=predicciones_test_c29.values,
    mode='lines+markers',
    name='Predicción Holt amortiguado',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Zoom: Test 2025 (Predicho vs Real)',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0,
        y=1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig2.show()

errores_test = test_c29.values - predicciones_test_c29.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses,
    y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple',
    marker_line_color='darkorchid',
    marker_line_width=1,
    opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(
    y=0,
    line_width=1.5,
    line_color='black',
    line_dash='solid',
    opacity=1
)

fig3.update_layout(
    title=dict(
        text='Errores por mes (Real - Predicción) - Test 2025',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig3.show()

## ARIMA

In [66]:
def evaluar_arima(train_fold, val_fold, order=(1,1,0)):
    """
    Ajusta ARIMA con el orden (p,d,q) dado.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = ARIMA(train_fold, order=order)
    modelo_ajustado = modelo.fit()
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [67]:
ordenes_arima = [
    ((1,1,0), 'ARIMA(1,1,0)'),
    ((1,1,1), 'ARIMA(1,1,1)'),
    ((0,1,1), 'ARIMA(0,1,1)'),   
    ((2,1,0), 'ARIMA(2,1,0)'),
    ((2,1,1), 'ARIMA(2,1,1)')    
]

resultados_cv_arima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos ARIMA (C29)")
print("-" * 50)

for orden, nombre in ordenes_arima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
        predicciones, met, _ = evaluar_arima(train_fold, val_fold, order=orden)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_arima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_arima = pd.DataFrame(resultados_cv_arima)
mejor_idx = df_cv_arima['RMSE'].idxmin()
mejor_orden, mejor_nombre = ordenes_arima[mejor_idx]

print("\n" + "-" * 50)
print("Mejor modelo ARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_arima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos ARIMA (C29)
--------------------------------------------------

--- ARIMA(1,1,0) ---
  Fold 1: RMSE=769.35, MAE=720.25, MAPE=27.95%, SMAPE=24.10%, MSE=591892.66
  Fold 2: RMSE=705.21, MAE=576.03, MAPE=20.07%, SMAPE=19.75%, MSE=497319.39
  Fold 3: RMSE=1191.20, MAE=1062.62, MAPE=43.14%, SMAPE=31.17%, MSE=1418960.52
  Fold 4: RMSE=787.72, MAE=585.53, MAPE=27.11%, SMAPE=21.16%, MSE=620501.55
  Promedio -> RMSE=863.37, MAE=736.11, MAPE=29.57%, SMAPE=24.04%, MSE=782168.53

--- ARIMA(1,1,1) ---
  Fold 1: RMSE=677.08, MAE=620.83, MAPE=24.17%, SMAPE=21.15%, MSE=458439.61
  Fold 2: RMSE=743.58, MAE=610.93, MAPE=20.38%, SMAPE=21.00%, MSE=552910.39
  Fold 3: RMSE=793.01, MAE=401.68, MAPE=21.93%, SMAPE=13.77%, MSE=628869.89
  Fold 4: RMSE=776.65, MAE=578.17, MAPE=26.59%, SMAPE=20.95%, MSE=603184.82
  Promedio -> RMSE=747.58, MAE=552.90, MAPE=23.27%, SMAPE=19.22%, MSE=560851.18

--- ARIMA(0,1,1) ---
  Fol

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
1,"ARIMA(1,1,1)",747.58,552.90,23.27,19.22,560851.18
4,"ARIMA(2,1,1)",776.90,585.79,25.24,20.06,605785.48
2,"ARIMA(0,1,1)",786.55,646.35,26.16,21.88,624517.42
3,"ARIMA(2,1,0)",820.46,685.98,27.71,22.86,687434.40
0,"ARIMA(1,1,0)",863.37,736.11,29.57,24.04,782168.53


In [68]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_arima = ARIMA(train_c29, order=mejor_orden)
modelo_final_arima_ajustado = modelo_final_arima.fit()

pred_test_arima = modelo_final_arima_ajustado.forecast(steps=len(test_c29))

real_test = test_c29.values
pred_test = pred_test_arima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

--------------------------------------------------
Evaluación final en Test (2025) - ARIMA(1,1,1)
--------------------------------------------------
RMSE: 804.70
MAE: 738.88
MAPE: 63.54%
SMAPE: 44.57%
MSE: 647538.70


In [69]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index,
    y=train_c29.values,
    mode='lines+markers',
    name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=test_c29.values,
    mode='lines+markers',
    name='Test Real (2025)',
    line=dict(color='green', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=pred_test_arima.values,
    mode='lines+markers',
    name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2025-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2025-01-01',
    y=1, 
    yref='paper',
    text='<b>Inicio Test</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text=f'C29 - {mejor_nombre}: Predicción vs Real<br>'
             f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses,
    y=test_c29.values,
    mode='lines+markers',
    name='Real',
    line=dict(color='green', width=2),
    marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses,
    y=pred_test_arima.values,
    mode='lines+markers',
    name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Zoom: Test 2025 (Predicho vs Real)',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0,
        y=1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig2.show()

errores_test = test_c29.values - pred_test_arima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses,
    y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple',
    marker_line_color='darkorchid',
    marker_line_width=1,
    opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(
    y=0,
    line_width=1.5,
    line_color='black',
    line_dash='solid',
    opacity=1
)

fig3.update_layout(
    title=dict(
        text='Errores por mes (Real - Predicción) - Test 2025',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig3.show()

## SARIMA

In [70]:
def evaluar_sarima(train_fold, val_fold, order=(1,1,1), seasonal_order=(0,0,0,12)):
    """
    Ajusta un modelo SARIMAX con los órdenes especificados.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = SARIMAX(
        train_fold,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,   
        enforce_invertibility=False
    )
    modelo_ajustado = modelo.fit(disp=False)
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [71]:
configuraciones_sarima = [
    ((1,1,1), (0,0,0,12), 'SARIMA(1,1,1)(0,0,0)[12]'),
    ((1,1,1), (1,0,0,12), 'SARIMA(1,1,1)(1,0,0)[12]'),
    ((1,1,1), (0,0,1,12), 'SARIMA(1,1,1)(0,0,1)[12]'),
    ((1,1,1), (1,0,1,12), 'SARIMA(1,1,1)(1,0,1)[12]'),
    ((0,1,1), (0,0,0,12), 'SARIMA(0,1,1)(0,0,0)[12]'),
    ((0,1,1), (1,0,0,12), 'SARIMA(0,1,1)(1,0,0)[12]'),
    ((0,1,1), (0,0,1,12), 'SARIMA(0,1,1)(0,0,1)[12]'),
    ((0,1,1), (1,0,1,12), 'SARIMA(0,1,1)(1,0,1)[12]'),
    ((1,1,0), (1,0,0,12), 'SARIMA(1,1,0)(1,0,0)[12]'),
    ((0,1,1), (0,1,1,12), 'SARIMA(0,1,1)(0,1,1)[12]'),
    ((1,1,1), (0,1,1,12), 'SARIMA(1,1,1)(0,1,1)[12]')
]

resultados_cv_sarima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos SARIMA (C29)")
print("-" * 50)

for order, seasonal_order, nombre in configuraciones_sarima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
        predicciones, met, _ = evaluar_sarima(
            train_fold, val_fold,
            order=order,
            seasonal_order=seasonal_order
        )
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_sarima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")


df_cv_sarima = pd.DataFrame(resultados_cv_sarima)
mejor_idx = df_cv_sarima['RMSE'].idxmin()
mejor_order, mejor_seasonal, mejor_nombre = configuraciones_sarima[mejor_idx]

print("\n" + "-" * 50)
print("Mejor modelo SARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_sarima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos SARIMA (C29)
--------------------------------------------------

--- SARIMA(1,1,1)(0,0,0)[12] ---
  Fold 1: RMSE=679.58, MAE=624.03, MAPE=24.31%, SMAPE=21.25%, MSE=461822.86
  Fold 2: RMSE=740.01, MAE=607.17, MAPE=20.33%, SMAPE=20.87%, MSE=547613.22
  Fold 3: RMSE=761.58, MAE=365.38, MAPE=20.39%, SMAPE=12.78%, MSE=580011.68
  Fold 4: RMSE=776.33, MAE=577.93, MAPE=26.57%, SMAPE=20.95%, MSE=602681.04
  Promedio -> RMSE=739.37, MAE=543.63, MAPE=22.90%, SMAPE=18.96%, MSE=548032.20

--- SARIMA(1,1,1)(1,0,0)[12] ---
  Fold 1: RMSE=544.59, MAE=504.37, MAPE=19.45%, SMAPE=17.70%, MSE=296574.08
  Fold 2: RMSE=774.71, MAE=616.21, MAPE=19.87%, SMAPE=21.20%, MSE=600173.49
  Fold 3: RMSE=824.35, MAE=529.94, MAPE=25.75%, SMAPE=17.46%, MSE=679555.48
  Fold 4: RMSE=772.39, MAE=586.53, MAPE=26.71%, SMAPE=21.26%, MSE=596584.38
  Promedio -> RMSE=729.01, MAE=559.26, MAPE=22.94%, SMAPE=19.40%, MSE=543221.85

--- 

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
1,"SARIMA(1,1,1)(1,0,0)[12]",7.290100e+02,5.592600e+02,2.294000e+01,19.40,5.432218e+05
0,"SARIMA(1,1,1)(0,0,0)[12]",7.393700e+02,5.436300e+02,2.290000e+01,18.96,5.480322e+05
6,"SARIMA(0,1,1)(0,0,1)[12]",7.693100e+02,6.433200e+02,2.568000e+01,21.77,6.148815e+05
4,"SARIMA(0,1,1)(0,0,0)[12]",8.032600e+02,6.668800e+02,2.692000e+01,22.40,6.551658e+05
5,"SARIMA(0,1,1)(1,0,0)[12]",8.148800e+02,6.832200e+02,2.691000e+01,22.66,7.038248e+05
8,"SARIMA(1,1,0)(1,0,0)[12]",8.443100e+02,7.386600e+02,2.852000e+01,24.16,7.773448e+05
9,"SARIMA(0,1,1)(0,1,1)[12]",9.693300e+02,7.560300e+02,2.957000e+01,26.32,1.000947e+06
10,"SARIMA(1,1,1)(0,1,1)[12]",9.787400e+02,8.055300e+02,3.171000e+01,26.50,1.047771e+06
3,"SARIMA(1,1,1)(1,0,1)[12]",1.739514e+09,1.734352e+09,6.165011e+07,63.79,1.210363e+19
2,"SARIMA(1,1,1)(0,0,1)[12]",8.613871e+10,8.613871e+10,3.065934e+09,64.61,2.967951e+22


In [72]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_sarima = SARIMAX(
    train_c29,
    order=mejor_order,
    seasonal_order=mejor_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)
modelo_final_sarima_ajustado = modelo_final_sarima.fit(disp=False)

pred_test_sarima = modelo_final_sarima_ajustado.forecast(steps=len(test_c29))

real_test = test_c29.values
pred_test = pred_test_sarima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

--------------------------------------------------
Evaluación final en Test (2025) - SARIMA(1,1,1)(1,0,0)[12]
--------------------------------------------------
RMSE: 740.59
MAE: 700.91
MAPE: 58.75%
SMAPE: 42.95%
MSE: 548479.12


In [73]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index,
    y=train_c29.values,
    mode='lines+markers',
    name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=test_c29.values,
    mode='lines+markers',
    name='Test Real (2025)',
    line=dict(color='green', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=pred_test_sarima.values,
    mode='lines+markers',
    name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2025-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2025-01-01',
    y=1, 
    yref='paper',
    text='<b>Inicio Test</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text=f'C29 - {mejor_nombre}: Predicción vs Real<br>'
             f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses,
    y=test_c29.values,
    mode='lines+markers',
    name='Real',
    line=dict(color='green', width=2),
    marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses,
    y=pred_test_sarima.values,
    mode='lines+markers',
    name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Zoom: Test 2025 (Predicho vs Real)',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0,
        y=1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig2.show()

errores_test = test_c29.values - pred_test_sarima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses,
    y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple',
    marker_line_color='darkorchid',
    marker_line_width=1,
    opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(
    y=0,
    line_width=1.5,
    line_color='black',
    line_dash='solid',
    opacity=1
)

fig3.update_layout(
    title=dict(
        text='Errores por mes (Real - Predicción) - Test 2025',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig3.show()

## Dynamic Optimized Theta

In [74]:

def evaluar_theta(train_fold, val_fold, theta=2.0, period=12):
    """
    Ajusta un modelo Theta con el parámetro theta dado.
    Procedimiento:
      1. Descomponer la serie con STL (period=12) para obtener estacionalidad.
      2. Serie desestacionalizada = serie - estacionalidad.
      3. Ajustar regresión lineal a la serie desestacionalizada (tendencia global).
      4. Ajustar SES a la serie desestacionalizada.
      5. Combinar pronósticos: Yhat = SES_forecast + theta*(trend_forecast - SES_forecast).
      6. Sumar estacionalidad (último año).
    Retorna predicciones, métricas y el modelo ajustado (un diccionario con componentes).
    """

    if len(train_fold) < 2 * period:
        ses = ExponentialSmoothing(train_fold, trend=None, seasonal=None,
                                   initialization_method='estimated').fit()
        pred = ses.forecast(len(val_fold))
        real = val_fold.values
        pred = pred.values
        mse = mean_squared_error(real, pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(real, pred)
        mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
        smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
        return pred, {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}, {'theta': theta, 'method': 'SES'}
    
    stl = STL(train_fold, period=period, robust=True).fit()
    seasonal = stl.seasonal  
    deseasonalized = train_fold - seasonal  

    x = np.arange(len(deseasonalized))
    slope, intercept, _, _, _ = sp_stats.linregress(x, deseasonalized.values)
    trend_line = slope * x + intercept

    ses_model = ExponentialSmoothing(deseasonalized, trend=None, seasonal=None,
                                     initialization_method='estimated').fit()
    
    h = len(val_fold)
    ses_forecast = ses_model.forecast(h)
    last_idx = len(deseasonalized) - 1
    trend_extrapolated = slope * np.arange(last_idx + 1, last_idx + 1 + h) + intercept

    theta_forecast = ses_forecast.values + theta * (trend_extrapolated - ses_forecast.values)


    last_year_seasonal = seasonal.iloc[-period:].values  
  
    repeats = int(np.ceil(h / period))
    seasonal_pattern = np.tile(last_year_seasonal, repeats)[:h]
    final_forecast = theta_forecast + seasonal_pattern

    real = val_fold.values
    pred = final_forecast

    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}
    modelo_info = {'theta': theta, 'slope': slope, 'intercept': intercept}
    
    return final_forecast, metricas, modelo_info

In [75]:
thetas = np.arange(0.0, 4.0 + 0.25, 0.25)  
resultados_cv_theta = []

print("-" * 50)
print("Validación cruzada - Optimización de Theta (C29)")
print("-" * 50)

for theta in thetas:
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
        _, met, _ = evaluar_theta(train_fold, val_fold, theta=theta, period=12)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_theta.append({
        'theta': theta,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })

df_cv_theta = pd.DataFrame(resultados_cv_theta)
mejor_idx = df_cv_theta['RMSE'].idxmin()
mejor_theta = df_cv_theta.loc[mejor_idx, 'theta']
mejor_rmse_cv = df_cv_theta.loc[mejor_idx, 'RMSE']

print("\n--- Resultados de optimización de theta ---")
print(f"Mejor theta: {mejor_theta:.2f} (RMSE CV promedio = {mejor_rmse_cv:.2f})")
print("\nTop 5 configuraciones:")
display(df_cv_theta.round(2).sort_values('RMSE').head())

print("\n" + "-" * 50)
print(f"Validación cruzada - Theta dinámico óptimo (θ={mejor_theta:.2f})")
print("-" * 50)

metricas_opt_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    _, met, _ = evaluar_theta(train_fold, val_fold, theta=mejor_theta, period=12)
    metricas_opt_folds['RMSE'].append(met['RMSE'])
    metricas_opt_folds['MAE'].append(met['MAE'])
    metricas_opt_folds['MAPE'].append(met['MAPE'])
    metricas_opt_folds['SMAPE'].append(met['SMAPE'])
    metricas_opt_folds['MSE'].append(met['MSE'])
    print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
          f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")

prom_opt = {k: np.mean(v) for k, v in metricas_opt_folds.items()}
print(f"  Promedio -> RMSE={prom_opt['RMSE']:.2f}, MAE={prom_opt['MAE']:.2f}, "
      f"MAPE={prom_opt['MAPE']:.2f}%, SMAPE={prom_opt['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Optimización de Theta (C29)
--------------------------------------------------

--- Resultados de optimización de theta ---
Mejor theta: 0.50 (RMSE CV promedio = 782.15)

Top 5 configuraciones:


,theta,RMSE,MAE,MAPE,SMAPE,MSE
2,0.50,782.15,638.18,24.42,23.77,613249.37
1,0.25,785.81,628.67,24.33,23.58,619594.57
3,0.75,799.30,669.57,25.21,24.71,640765.08
0,0.00,810.70,630.02,24.60,23.76,659800.70
4,1.00,835.14,712.80,26.41,26.10,702141.71



--------------------------------------------------
Validación cruzada - Theta dinámico óptimo (θ=0.50)
--------------------------------------------------
  Fold 1: RMSE=829.47, MAE=717.91, MAPE=27.08%, SMAPE=27.35%, MSE=688026.40
  Fold 2: RMSE=804.34, MAE=706.67, MAPE=22.95%, SMAPE=24.97%, MSE=646956.14
  Fold 3: RMSE=727.04, MAE=549.36, MAPE=23.23%, SMAPE=20.45%, MSE=528587.40
  Fold 4: RMSE=767.74, MAE=578.79, MAPE=24.41%, SMAPE=22.32%, MSE=589427.53
  Promedio -> RMSE=782.15, MAE=638.18, MAPE=24.42%, SMAPE=23.77%, MSE=4079813.03


In [76]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - Theta optimizado (θ={mejor_theta:.2f})")
print("-" * 50)

pred_test_theta, met_train, modelo_info = evaluar_theta(train_c29, test_c29, theta=mejor_theta, period=12)

real_test = test_c29.values
pred_test = pred_test_theta

mse_test = met_train['MSE']
rmse_test = met_train['RMSE']
mae_test = met_train['MAE']
mape_test = met_train['MAPE']
smape_test = met_train['SMAPE']

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c29.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Theta optimizado (θ=0.50)
--------------------------------------------------
RMSE: 1513.74
MAE: 1445.36
MAPE: 107.29%
SMAPE: 68.45%
MSE: 2291415.38


In [77]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index,
    y=train_c29.values,
    mode='lines+markers',
    name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=test_c29.values,
    mode='lines+markers',
    name='Test Real (2025)',
    line=dict(color='green', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=pred_test_theta,
    mode='lines+markers',
    name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2025-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2025-01-01',
    y=1, 
    yref='paper',
    text='<b>Inicio Test</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text=f'C29 - Theta Dinámico Optimizado (θ={mejor_theta:.2f}): Predicción vs Real<br>'
             f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses,
    y=test_c29.values,
    mode='lines+markers',
    name='Real',
    line=dict(color='green', width=2),
    marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses,
    y=pred_test_theta,
    mode='lines+markers',
    name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Zoom: Test 2025 (Predicho vs Real)',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0,
        y=1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig2.show()

errores_test = test_c29.values - pred_test_theta

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses,
    y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple',
    marker_line_color='darkorchid',
    marker_line_width=1,
    opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(
    y=0,
    line_width=1.5,
    line_color='black',
    line_dash='solid',
    opacity=1
)

fig3.update_layout(
    title=dict(
        text='Errores por mes (Real - Predicción) - Test 2025',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig3.show()

## Ridge

In [78]:
def crear_dataset_supervisado(serie, n_input=12, n_output=12):
    """
    Convierte una serie univariada en matrices X, y para
    predicción multi‑output directa de horizonte fijo.
    Cada muestra: X = [t-n_input : t], y = [t : t+n_output].
    """
    X, y = [], []
    for i in range(len(serie) - n_input - n_output + 1):
        X.append(serie[i:i + n_input])
        y.append(serie[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_ridge(train_fold, val_fold, n_input=12, alphas=None):
    """
    Entrena Ridge con selección de alpha vía TimeSeriesSplit
    sobre los datos generados del train_fold.
    Retorna predicciones (12 meses), métricas y el modelo ajustado.
    """
    if alphas is None:
        alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    
    X_train, y_train = crear_dataset_supervisado(train_fold.values, 
                                                 n_input=n_input, 
                                                 n_output=len(val_fold))
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = RidgeCV(alphas=alphas, cv=tscv)
    modelo.fit(X_train_scaled, y_train)
    
    X_pred = train_fold.values[-n_input:].reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [79]:
resultados_cv_ridge = []

print("-" * 50)
print("Validación cruzada - Ridge Regression (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_ridge(train_fold, val_fold)
    
    resultados_cv_ridge.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_ridge = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_ridge]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_ridge]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_ridge]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_ridge]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_ridge])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_ridge['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_ridge['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_ridge['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_ridge['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_ridge['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Ridge Regression (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 10.00
RMSE: 1658.40
MAE: 1579.89
MAPE: 59.76%
SMAPE: 44.80%
MSE: 2750282.12

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 1000.00
RMSE: 831.28
MAE: 657.59
MAPE: 25.44%
SMAPE: 21.86%
MSE: 691023.25

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 1000.00
RMSE: 610.03
MAE: 331.26
MAPE: 17.14%
SMAPE: 12.23%
MSE: 372142.37

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 1000.00
RMSE: 704.94
MAE: 545.36
MAPE: 24.19%
SMAPE: 19.99%
MSE: 496934.88

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 951.16
MAE: 778.52
MAPE: 31.63%
SMAPE: 24.72%
MSE: 1077595.66


In [80]:
print("-" * 50)
print("Evaluación final en Test (2025) - Ridge Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado(
    train_c29.values, 
    n_input=12, 
    n_output=12
)

scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
modelo_final_ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0], cv=tscv)
modelo_final_ridge.fit(X_train_scaled, y_train_full)

print(f"Alpha seleccionado: {modelo_final_ridge.alpha_:.2f}")

X_pred_test = train_c29.values[-12:].reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)
pred_test_ridge = modelo_final_ridge.predict(X_pred_test_scaled).flatten()

real_test = test_c29.values
pred_test = pred_test_ridge

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c29.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Ridge Regression
--------------------------------------------------
Alpha seleccionado: 1000.00
RMSE: 1724.24
MAE: 1704.32
MAPE: 137.52%
SMAPE: 78.43%
MSE: 2973008.36


In [81]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index, y=train_c29.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=test_c29.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C29 - Ridge Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c29.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c29.values - pred_test_ridge

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Lasso

In [82]:
def crear_dataset_supervisado_lasso(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con los rezagos y características determinísticas:
    - Tendencia lineal normalizada
    - Componentes cíclicos del mes (seno/coseno)
    - Dummy COVID
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lasso(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta MultiTaskLassoCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = np.logspace(-4, 4, 50)  
    
    X_train, y_train = crear_dataset_supervisado_lasso(
        train_fold.values, 
        train_fold.index, 
        n_input=n_input, 
        n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train[:, :n_input] = scaler.fit_transform(X_train[:, :n_input])
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
    modelo.fit(X_train, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = len(train_fold) / len(train_fold)
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred[:, :n_input] = scaler.transform(X_pred[:, :n_input])
    
    predicciones = modelo.predict(X_pred).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [83]:
resultados_cv_lasso = []

print("-" * 50)
print("Validación cruzada - Lasso Regression (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lasso(train_fold, val_fold)
    
    resultados_cv_lasso.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lasso = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lasso]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lasso]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lasso]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lasso]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lasso])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lasso['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lasso['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lasso['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lasso['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lasso['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Lasso Regression (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 719.69
RMSE: 1349.40
MAE: 1280.39
MAPE: 48.72%
SMAPE: 38.18%
MSE: 1820887.07

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 719.69
RMSE: 936.28
MAE: 772.10
MAPE: 29.82%
SMAPE: 25.04%
MSE: 876616.78

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 10000.00
RMSE: 618.39
MAE: 334.24
MAPE: 17.35%
SMAPE: 12.30%
MSE: 382409.73

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 10000.00
RMSE: 701.53
MAE: 540.16
MAPE: 24.02%
SMAPE: 19.82%
MSE: 492140.79

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 901.40
MAE: 731.72
MAPE: 29.98%
SMAPE: 23.84%
MSE: 893013.59


In [84]:
print("-" * 50)
print("Evaluación final en Test (2025) - Lasso Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lasso(
    train_c29.values,
    train_c29.index,
    n_input=12,
    n_output=12
)

scaler_final = StandardScaler()
X_train_full[:, :12] = scaler_final.fit_transform(X_train_full[:, :12])

tscv = TimeSeriesSplit(n_splits=3)
modelo_final_lasso = MultiTaskLassoCV(alphas=np.logspace(-4, 4, 50), cv=tscv, max_iter=20000, random_state=42)
modelo_final_lasso.fit(X_train_full, y_train_full)
print(f"Alpha seleccionado: {modelo_final_lasso.alpha_:.2f}")

x_lags_pred_test = train_c29.values[-12:]
fecha_fin_pred_test = train_c29.index[-1]
tendencia_pred_test = len(train_c29) / len(train_c29)
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test[:, :12] = scaler_final.transform(X_pred_test[:, :12])

pred_test_lasso = modelo_final_lasso.predict(X_pred_test).flatten()

real_test = test_c29.values
pred_test = pred_test_lasso

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c29.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Lasso Regression
--------------------------------------------------
Alpha seleccionado: 10000.00
RMSE: 1769.77
MAE: 1753.09
MAPE: 140.95%
SMAPE: 79.78%
MSE: 3132083.80


In [85]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index, y=train_c29.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=test_c29.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C29 - Lasso Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c29.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c29.values - pred_test_lasso

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Random Forest

In [86]:
def crear_dataset_supervisado_rf(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_random_forest(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta Random Forest multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler (aunque RF no necesita escalado).
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [5, 10, 15, None],
            'estimator__min_samples_split': [2, 5, 10]
        }
    
    X_train, y_train = crear_dataset_supervisado_rf(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    rf = RandomForestRegressor(random_state=42)
    multi_rf = MultiOutputRegressor(rf)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_rf, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [87]:
resultados_cv_rf = []

print("-" * 50)
print("Validación cruzada - Random Forest (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_random_forest(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    min_samples_split = best_params['estimator__min_samples_split']
    
    resultados_cv_rf.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_rf = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_rf]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_rf]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_rf]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_rf]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_rf])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_rf['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_rf['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_rf['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_rf['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_rf['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Random Forest (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=300, max_depth=10, min_samples_split=2
RMSE: 1091.36
MAE: 1030.46
MAPE: 39.37%
SMAPE: 32.11%
MSE: 1191070.57

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=200, max_depth=10, min_samples_split=2
RMSE: 816.05
MAE: 633.10
MAPE: 24.07%
SMAPE: 21.29%
MSE: 665932.32

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=10, min_samples_split=10
RMSE: 668.64
MAE: 500.24
MAPE: 22.17%
SMAPE: 18.11%
MSE: 447076.69

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=200, max_depth=10, min_samples_split=10
RMSE: 811.14
MAE: 644.39
MAPE: 28.05%
SMAPE: 23.15%
MSE: 657949.71

---------------------------------------

In [88]:
print("-" * 50)
print("Evaluación final en Test (2025) - Random Forest")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_rf(
    train_c29.values, train_c29.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [5, 10, 15, None],
    'estimator__min_samples_split': [2, 5, 10]
}

rf_final = RandomForestRegressor(random_state=42)
multi_rf_final = MultiOutputRegressor(rf_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_rf_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_rf = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c29.values[-12:]
fecha_fin_pred_test = train_c29.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0  # 2025 no tiene COVID
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_rf = modelo_final_rf.predict(X_pred_test_scaled).flatten()

real_test = test_c29.values
pred_test = pred_test_rf

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c29.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Random Forest
--------------------------------------------------
Mejores parámetros: {'estimator__max_depth': 10, 'estimator__min_samples_split': 10, 'estimator__n_estimators': 200}
RMSE: 1671.37
MAE: 1647.96
MAPE: 130.34%
SMAPE: 76.72%
MSE: 2793463.64


In [89]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index, y=train_c29.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=test_c29.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C29 - Random Forest: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c29.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c29.values - pred_test_rf

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## XGBoost

In [90]:
def crear_dataset_supervisado_xgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas:
    - 12 rezagos
    - Tendencia normalizada
    - Componentes cíclicos del mes (seno y coseno)
    - Dummy COVID
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_xgboost(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta XGBoost multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_xgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    xgb = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
    multi_xgb = MultiOutputRegressor(xgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_xgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0  # última posición
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [91]:
resultados_cv_xgb = []

print("-" * 50)
print("Validación cruzada - XGBoost (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_xgboost(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_xgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_xgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_xgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_xgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_xgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_xgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_xgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_xgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_xgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_xgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_xgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_xgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - XGBoost (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 1122.29
MAE: 1053.44
MAPE: 40.13%
SMAPE: 32.63%
MSE: 1259531.25

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=5, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 873.21
MAE: 704.42
MAPE: 26.84%
SMAPE: 23.25%
MSE: 762487.56

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=5, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 555.11
MAE: 358.40
MAPE: 17.22%
SMAPE: 13.32%
MSE: 308143.19

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=5, learning_rate=0.01, subsample=0

In [92]:
print("-" * 50)
print("Evaluación final en Test (2025) - XGBoost")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_xgb(
    train_c29.values, train_c29.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, 9],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

xgb_final = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
multi_xgb_final = MultiOutputRegressor(xgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_xgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_xgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c29.values[-12:]
fecha_fin_pred_test = train_c29.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0  # 2025 no tiene COVID
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_xgb = modelo_final_xgb.predict(X_pred_test_scaled).flatten()

real_test = test_c29.values
pred_test = pred_test_xgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c29.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - XGBoost
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 5, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 1774.74
MAE: 1758.69
MAPE: 140.33%
SMAPE: 79.93%
MSE: 3149689.25


In [93]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index, y=train_c29.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=test_c29.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C29 - XGBoost: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c29.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c29.values - pred_test_xgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## LightGBM

In [94]:
def crear_dataset_supervisado_lgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lightgbm(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta LightGBM multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7, -1],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_lgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb = LGBMRegressor(random_state=42, verbosity=-1)
    multi_lgb = MultiOutputRegressor(lgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_lgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [95]:
resultados_cv_lgb = []

print("-" * 50)
print("Validación cruzada - LightGBM (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lightgbm(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_lgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

# Métricas promedio
metricas_promedio_lgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - LightGBM (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 916.52
MAE: 833.37
MAPE: 31.97%
SMAPE: 26.75%
MSE: 840007.24

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 822.60
MAE: 648.11
MAPE: 25.08%
SMAPE: 21.57%
MSE: 676674.18

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 618.39
MAE: 334.24
MAPE: 17.35%
SMAPE: 12.30%
MSE: 382409.73

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8

In [96]:
print("-" * 50)
print("Evaluación final en Test (2025) - LightGBM")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lgb(
    train_c29.values, train_c29.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, -1],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

lgb_final = LGBMRegressor(random_state=42, verbosity=-1)
multi_lgb_final = MultiOutputRegressor(lgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_lgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_lgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c29.values[-12:]
fecha_fin_pred_test = train_c29.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lgb = modelo_final_lgb.predict(X_pred_test_scaled).flatten()

real_test = test_c29.values
pred_test = pred_test_lgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c29.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - LightGBM
--------------------------------------------------


Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 3, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 1659.20
MAE: 1638.42
MAPE: 131.95%
SMAPE: 76.61%
MSE: 2752949.51


In [97]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index, y=train_c29.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=test_c29.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C29 - LightGBM: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c29.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c29.values - pred_test_lgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## KNN

In [98]:
def crear_dataset_supervisado_knn(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_knn(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta KNN multi‑output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_neighbors': [2, 3, 5, 7, 10],
            'estimator__weights': ['uniform', 'distance'],
            'estimator__p': [1, 2]   
        }
    
    X_train, y_train = crear_dataset_supervisado_knn(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    knn = KNeighborsRegressor()
    multi_knn = MultiOutputRegressor(knn)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_knn, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [99]:
resultados_cv_knn = []

print("-" * 50)
print("Validación cruzada - KNN Regression (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_knn(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_neighbors = best_params['estimator__n_neighbors']
    weights = best_params['estimator__weights']
    p = best_params['estimator__p']
    
    resultados_cv_knn.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_neighbors': n_neighbors,
        'weights': weights,
        'p': p,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_neighbors={n_neighbors}, weights={weights}, p={p}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_knn = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_knn]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_knn]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_knn]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_knn]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_knn])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_knn['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_knn['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_knn['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_knn['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_knn['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - KNN Regression (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_neighbors=3, weights=distance, p=2
RMSE: 1529.27
MAE: 1467.87
MAPE: 55.71%
SMAPE: 42.60%
MSE: 2338656.57

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_neighbors=7, weights=distance, p=2
RMSE: 766.49
MAE: 579.37
MAPE: 21.44%
SMAPE: 19.56%
MSE: 587509.04

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_neighbors=10, weights=uniform, p=1
RMSE: 507.90
MAE: 290.74
MAPE: 14.61%
SMAPE: 11.10%
MSE: 257961.41

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_neighbors=10, weights=uniform, p=2
RMSE: 793.82
MAE: 692.57
MAPE: 28.68%
SMAPE: 24.98%
MSE: 630151.74

--------------------------------------------------
Métricas promedio en validación cruzada
---------

In [100]:
print("-" * 50)
print("Evaluación final en Test (2025) - KNN Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_knn(
    train_c29.values, train_c29.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_neighbors': [2, 3, 5, 7, 10, 12],
    'estimator__weights': ['uniform', 'distance'],
    'estimator__p': [1, 2]
}

knn_final = KNeighborsRegressor()
multi_knn_final = MultiOutputRegressor(knn_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_knn_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_knn = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c29.values[-12:]
fecha_fin_pred_test = train_c29.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_knn = modelo_final_knn.predict(X_pred_test_scaled).flatten()

real_test = test_c29.values
pred_test = pred_test_knn

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c29.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - KNN Regression
--------------------------------------------------
Mejores parámetros: {'estimator__n_neighbors': 12, 'estimator__p': 1, 'estimator__weights': 'uniform'}
RMSE: 1622.15
MAE: 1593.19
MAPE: 128.35%
SMAPE: 75.19%
MSE: 2631373.26


In [101]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index, y=train_c29.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=test_c29.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C29 - KNN Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c29.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c29.values - pred_test_knn

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Tabla 1 – Métricas Promedio en Validación Cruzada (CV)

| Modelo                     | RMSE      | MAE      | MAPE    | SMAPE   | MSE         |
|----------------------------|-----------|----------|---------|---------|-------------|
| Holt‑Winters               | 792.78    | 649.97   | 25.97%  | 22.01%  | 636726.65   |
| ARIMA(1,1,1)               | 747.58    | 552.90   | 23.27%  | 19.22%  | 560851.18   |
| SARIMA(1,1,1)(1,0,0)[12]   | 729.01    | 559.26   | 22.94%  | 19.40%  | 543221.85   |
| Theta (θ=0.50)             | 782.15    | 638.18   | 24.42%  | 23.77%  | 4079813.03  |
| Ridge                      | 951.16    | 778.52   | 31.63%  | 24.72%  | 1077595.66  |
| Lasso                      | 901.40    | 731.72   | 29.98%  | 23.84%  | 893013.59   |
| Random Forest              | 846.80    | 702.05   | 28.41%  | 23.67%  | 740507.32   |
| XGBoost                    | 835.56    | 678.32   | 27.79%  | 22.69%  | 739211.48   |
| LightGBM                   | 772.18    | 603.85   | 24.99%  | 20.59%  | 608431.26   |
| KNN                        | 899.37    | 757.64   | 30.11%  | 24.56%  | 953569.69   |

## Tabla 2 – Métricas en Test (2025)

| Modelo                     | RMSE      | MAE      | MAPE    | SMAPE   | MSE         |
|----------------------------|-----------|----------|---------|---------|-------------|
| Holt‑Winters               | 479.43    | 405.89   | 36.52%  | 28.19%  | 229851.11   |
| ARIMA(1,1,1)               | 804.70    | 738.88   | 63.54%  | 44.57%  | 647538.70   |
| SARIMA(1,1,1)(1,0,0)[12]   | 740.59    | 700.91   | 58.75%  | 42.95%  | 548479.12   |
| Theta (θ=0.50)             | 1513.74   | 1445.36  | 107.29% | 68.45%  | 2291415.38  |
| Ridge                      | 1724.24   | 1704.32  | 137.52% | 78.43%  | 2973008.36  |
| Lasso                      | 1769.77   | 1753.09  | 140.95% | 79.78%  | 3132083.80  |
| Random Forest              | 1671.37   | 1647.96  | 130.34% | 76.72%  | 2793463.64  |
| XGBoost                    | 1774.74   | 1758.69  | 140.33% | 79.93%  | 3149689.25  |
| LightGBM                   | 1659.20   | 1638.42  | 131.95% | 76.61%  | 2752949.51  |
| KNN                        | 1622.15   | 1593.19  | 128.35% | 75.19%  | 2631373.26  |

## Tabla 3 – Errores por Mes en Test (2025)

| Modelo                     | Ene   | Feb   | Mar  | Abr   | May   | Jun   | Jul   | Ago   | Sep   | Oct   | Nov   | Dic   |
|----------------------------|-------|-------|------|-------|-------|-------|-------|-------|-------|-------|-------|-------|
| Holt‑Winters               | -86   | -513  | 39   | -93   | -273  | -261  | -386  | -567  | -533  | -560  | -936  | -652  |
| ARIMA(1,1,1)               | -235  | -777  | -261 | -411  | -606  | -607  | -747  | -941  | -920  | -961  | -1350 | -1052 |
| SARIMA(1,1,1)(1,0,0)[12]   | -250  | -766  | -356 | -415  | -666  | -692  | -771  | -919  | -827  | -818  | -1130 | -800  |
| Theta (θ=0.50)             | -1207 | -1618 | -1712| -1499 | -1872 | -2045 | -1908 | -1719 | -1346 | -1025 | -951  | -442  |
| Ridge                      | -1370 | -1808 | -1272| -1389 | -1574 | -1594 | -1724 | -1900 | -1848 | -1863 | -2215 | -1894 |
| Lasso                      | -1475 | -1894 | -1349| -1459 | -1631 | -1630 | -1760 | -1927 | -1871 | -1889 | -2243 | -1910 |
| Random Forest              | -1135 | -1874 | -1142| -1508 | -1774 | -1868 | -1894 | -1959 | -1750 | -1728 | -1786 | -1356 |
| XGBoost                    | -1395 | -1935 | -1273| -1535 | -1808 | -1782 | -1877 | -1942 | -1924 | -1849 | -2126 | -1657 |
| LightGBM                   | -1249 | -1739 | -1213| -1333 | -1436 | -1593 | -1792 | -1910 | -1855 | -1713 | -2058 | -1769 |
| KNN                        | -1215 | -1434 | -896 | -1314 | -1633 | -1751 | -1806 | -1892 | -1821 | -1694 | -1987 | -1677 |

## Interpretación de los Resultados de los Modelos

**a) Desempeño en validación cruzada (CV, 2018‑2024)**
- Los modelos que mejor ajustaron el período histórico fueron SARIMA(1,1,1)(1,0,0)[12] y ARIMA(1,1,1), con RMSE promedios de 729 y 748 comparendos respectivamente y MAPE cercanas al 23%.  
- Holt‑Winters con tendencia amortiguada mostró un RMSE de 793, ligeramente superior, mientras que los métodos de machine learning (LightGBM, XGBoost, Random Forest) y los regularizados (Ridge, Lasso) obtuvieron errores similares o mayores en CV (772–951 RMSE).  
- Esto indica que, durante el período de entrenamiento, las estructuras autorregresivas y de media móvil capturaron eficazmente la autocorrelación residual detectada en la descomposición STL (Ljung‑Box significativo), aprovechando la tendencia y la débil estacionalidad.

**b) Desempeño en el conjunto de prueba (2025)**
- Todos los modelos, salvo Holt‑Winters, fracasaron de forma contundente en 2025. Los errores se dispararon a RMSE de 740–1775, MAPE de 58% a >140%, con subestimaciones masivas (errores negativos de magnitud entre 1000 y 2200 comparendos por mes).  
- Únicamente Holt‑Winters con tendencia amortiguada logró un RMSE de 479 y MAE de 406, con errores mensuales mucho más pequeños (ej. -86 en enero, -936 en noviembre). Aunque también subestimó los valores reales, su error fue de 2 a 4 veces menor que el de cualquier otro modelo.  
- Este contraste revela que el año 2025 presentó una aceleración de la tendencia decreciente no anticipada por los modelos entrenados con datos hasta 2024.

---

## ¿Por qué se llegó a estos resultados?

**a) Características de la serie y ajuste de los modelos**
- La descomposición STL confirmó una tendencia negativa significativa (-18.9 comparendos/mes, R²=0.58) y una estacionalidad muy débil (5.66% de la varianza). Los residuos no eran ruido blanco, sino que conservaban autocorrelación significativa (Ljung‑Box rechaza independencia) sin llegar a ser normales.  
- Por ello, en CV los modelos ARIMA(1,1,1) y SARIMA(1,1,1)(1,0,0)[12] resultaron óptimos: aplican una diferenciación para eliminar la tendencia y modelan la dependencia serial restante (AR y MA). Su drift constante estimaba una reducción promedio similar a la histórica.

**b) El quiebre de 2025**
- En 2025 la serie tocó valores mínimos históricos (858 comparendos en diciembre, cuando los mínimos previos rondaban 1350). La caída fue mucho más pronunciada que la tendencia media del período 2018‑2024.  
- Los modelos ARIMA/SARIMA, al proyectar un drift promedio constante, sobreestimaron enormemente la cantidad de infracciones. Los modelos de machine learning (Random Forest, XGBoost, LightGBM, KNN) y los regularizados (Ridge, Lasso) también fallaron porque extrapolaron patrones del pasado sin capacidad para adaptarse a una dinámica nunca vista en el histórico.  
- El modelo Theta optimizado y los basados en árboles sufrieron el mismo problema: cualquier estructura que dependiera de la “normalidad” histórica quedó invalidada por el cambio estructural de 2025.

**c) ¿Por qué Holt‑Winters amortiguado funcionó mejor?**
- La tendencia amortiguada de Holt no supone una disminución lineal infinita, sino que frena progresivamente la tendencia proyectándola hacia un nivel asintótico. Como la caída reciente era muy agresiva, el amortiguamiento generó pronósticos más bajos, acercándose a los valores reales de 2025.  
- Aun así, subestimó la caída (todos los errores son negativos), porque la realidad se desplomó aún más rápido de lo que el modelo podía asimilar a partir del suavizado exponencial. Pero fue la única aproximación capaz de reflejar, al menos en parte, la aceleración reciente.

**d) Residuos no normales y outliers**
- La presencia de residuos no normales y valores atípicos (picos en 2019 y caídas por pandemia en 2020) indica que la serie sufrió shocks externos. Esto explica que los modelos con supuestos más estrictos (ARIMA con normalidad implícita en la estimación) pudieran haber sido menos robustos frente al nuevo shock de 2025.

---

## Modelo Ideal para Predecir 2026

Con base en la evidencia, el modelo más adecuado para pronosticar 2026 es Holt‑Winters con tendencia amortiguada.

**Justificación:**  
- Fue el único que ofreció un desempeño aceptable en el período más reciente (test 2025), donde se materializó un cambio de régimen hacia una caída más acelerada.  
- Su mecánica de amortiguamiento evita extrapolar una reducción lineal perpetua, lo cual es realista: no se espera que las infracciones desciendan indefinidamente, sino que eventualmente se estabilicen.  
- Además, captura la débil estacionalidad anual presente en la serie.
- Aunque Holt‑Winters fue el mejor, todavía subestimó los valores reales (errores de hasta -936 en noviembre). Por tanto, los pronósticos para 2026 deben ser considerados como un escenario base moderado; la caída podría continuar si persisten las causas del desplome (políticas de control, menor movilidad, etc.).  

## Pronóstico de la Infracción C29 para el año 2026

Una vez seleccionado el mejor modelo predictivo (Holt-Winters con tendencia aditiva, tendencia amortiguada e inicialización estimada, sin componente estacional), se procede a realizar el pronóstico del volumen de infracciones por exceso de velocidad para los 12 meses del año 2026. El modelo se entrena con la serie completa de 96 meses (enero 2018 a diciembre 2025) y genera predicciones puntuales para cada mes de 2026. Los resultados se visualizan mediante dos gráficos interactivos: el primero muestra la serie histórica completa junto con la proyección hacia 2026, con una línea vertical discontinua que marca el inicio del período pronosticado; el segundo presenta exclusivamente el pronóstico mensual para 2026, facilitando la interpretación de la tendencia esperada mes a mes. Esta visualización permite anticipar el comportamiento futuro de la infracción y apoyar la toma de decisiones en materia de control y gestión vial.

In [102]:
modelo_final_2026 = ExponentialSmoothing(
    serie_c29,                
    trend='add',
    seasonal=None,
    damped_trend=True,
    initialization_method='estimated'
).fit()

forecast_2026 = modelo_final_2026.forecast(12)

idx_forecast = pd.date_range(start='2026-01-01', periods=12, freq='MS')
forecast_2026.index = idx_forecast

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=serie_c29.index,
    y=serie_c29.values,
    mode='lines+markers',
    name='Histórico (2018-2025)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=forecast_2026.index,
    y=forecast_2026.values,
    mode='lines+markers',
    name='Predicción 2026',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2026-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2026-01-01',
    y=1,
    yref='paper',
    text='<b>Inicio 2026</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text='C29 - Holt con tendencia amortiguada: Predicción 2026<br>'
             '<sup>Modelo entrenado con datos hasta diciembre 2025</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

meses_2026 = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
              'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses_2026,
    y=forecast_2026.values,
    mode='lines+markers',
    name='Predicción 2026',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Pronóstico C29 para el año 2026',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig2.show()